<a href="https://colab.research.google.com/github/NamishBansal15/substation-detection/blob/main/inference-mapping/model-based-prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import pandas as pd
import os
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')


# =========================
# PATHS
# =========================
IMAGE_DIR = "/content/drive/MyDrive/dataset-inference/images/naip_images/"
MODEL_PATH = "/content/drive/MyDrive/april-detection/yolo/yolov8s/weights/best.pt"
OUTPUT_CSV = "/content/drive/MyDrive/dataset-inference/data/component_predictions.csv"


# =========================
# LOAD MODEL (GPU 🔥)
# =========================
model = YOLO(MODEL_PATH)

# force GPU if available
DEVICE = 0  # use "cpu" if needed


# =========================
# PRELOAD IMAGE PATHS
# =========================
image_files = [
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.endswith(".png")
]


# =========================
# BATCH INFERENCE 🔥
# =========================
BATCH_SIZE = 16  # 🔥 try 8, 16, 32 depending on memory

results_data = []

for i in tqdm(range(0, len(image_files), BATCH_SIZE)):
    batch_paths = image_files[i:i + BATCH_SIZE]

    results = model(
        batch_paths,
        device=DEVICE,
        verbose=False
    )

    for img_path, result in zip(batch_paths, results):
        filename = os.path.basename(img_path)

        counts = {}

        # count detections
        for box in result.boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            counts[cls_name] = counts.get(cls_name, 0) + 1

        # extract id
        idx = int(filename.split("_")[1].split(".")[0])

        results_data.append({
            "id": idx,
            "image": filename,
            **counts
        })


# =========================
# SAVE OUTPUT
# =========================
df = pd.DataFrame(results_data).fillna(0)
df.to_csv(OUTPUT_CSV, index=False)

print("✅ Saved predictions!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 693/693 [31:33<00:00,  2.73s/it]


✅ Saved predictions!
